# GBDT vs NN

Thin orchestration notebook for the default `NN` vs `GBDT` analysis.

In [1]:
from __future__ import annotations

import sys
from pathlib import Path

notebook_dir = Path.cwd()
project_dir = notebook_dir.parent
repo_root = project_dir.parent

sys.path.insert(0, str(project_dir / "src"))
sys.path.insert(0, str(repo_root / "tabarena" / "tabarena"))

In [2]:
from mfa import load_config, run_analysis

config = load_config(project_dir / "configs" / "config_5.yaml")
result = run_analysis(
    config,
    # datasets=[
    #     "APSFailure",
    #     "GiveMeSomeCredit",
    #     # "Marketing_Campaign",
    #     # "bank-marketing",
    #     # "blood-transfusion-service-center",
    #     # "diabetes",
    #     # "maternal_health_risk",
    # ],
)
result.analysis_table.head()

09:15:39 INFO mfa.pipeline: Starting analysis: comparisons=tabpfn_old_vs_tabicl_old, tabpfn_new_vs_tabicl_new; scope=38 selected dataset(s); unit=dataset; method_variant=default,tuned; n_jobs=32
09:15:39 INFO mfa.pipeline: Stage 1/5 raw results: cache hit (24840 rows, 38 dataset(s))
09:15:39 INFO mfa.pipeline: Stage 2/5 meta-features: trace enabled; metafeature caches remain active, so live per-split diagnostics appear only on cache misses
09:15:39 INFO mfa.pipeline: Stage 2/5 meta-features: cache hit (594 rows, 38 dataset(s))
09:15:39 INFO mfa.pipeline: Stage 3/5 pairwise gaps: cache hit (1080 rows, 38 dataset(s))
09:15:39 INFO mfa.pipeline: Stage 4/5 analysis table: joining and aggregating at dataset level
/work/mherre/tabular-meta-feature-analysis/meta-feature-analysis/src/mfa/aggregation.py:121: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using

,dataset,comparison_name,group_a_name,group_b_name,group_a_label,group_b_label,expected_direction,n_splits,n,d,...,best_a_error,best_a_norm_error,best_b_error,best_b_norm_error,delta_raw,delta_raw_std,delta_raw_sem,delta_norm,delta_norm_std,delta_norm_sem
0,APSFailure,tabpfn_new_vs_tabicl_new,tabpfn_new,tabicl_new,tabpfn-new,tabicl-new,NaN,9,50666.666667,170.0,...,0.007312,0.481520,0.005776,0.000000,0.001537,0.000573,0.000191,0.481520,0.168284,0.056095
1,Amazon_employee_access,tabpfn_new_vs_tabicl_new,tabpfn_new,tabicl_new,tabpfn-new,tabicl-new,NaN,9,21846.000000,9.0,...,0.147324,0.801420,0.146974,0.789150,0.000351,0.003778,0.001259,0.012270,0.099236,0.033079
2,Bank_Customer_Churn,tabpfn_new_vs_tabicl_new,tabpfn_new,tabicl_new,tabpfn-new,tabicl-new,NaN,9,6666.666667,10.0,...,0.127787,0.240931,0.128107,0.265143,-0.000320,0.001064,0.000355,-0.024211,0.094995,0.031665
3,Bioresponse,tabpfn_new_vs_tabicl_new,tabpfn_new,tabicl_new,tabpfn-new,tabicl-new,NaN,9,2500.666667,1776.0,...,0.122321,0.153651,0.138276,1.000000,-0.015955,0.003924,0.001308,-0.846349,0.146674,0.048891
4,Diabetes130US,tabpfn_new_vs_tabicl_new,tabpfn_new,tabicl_new,tabpfn-new,tabicl-new,NaN,9,47678.666667,47.0,...,0.331797,0.277703,0.336171,0.564424,-0.004374,0.002479,0.000826,-0.286720,0.168534,0.056178


In [3]:
import pandas as pd

# -- Inspect what the result object contains --
print(f"config_hash:        {result.config_hash}")
print(f"comparison_name:    {result.comparison_name}")
print(f"analysis_table:     {result.analysis_table.shape}")
print(f"gap_table:          {result.gap_table.shape}")
print(f"metafeature_table:  {result.metafeature_table.shape}")
print(f"correlation_results: {len(result.correlation_results)} features tested")
print(
    f"correction_result:  {result.correction_result.method if result.correction_result else None}"
)
print(f"multivariate_result: {result.multivariate_result}")

config_hash:        740e427e068cd107
comparison_name:    None
analysis_table:     (64, 129)
gap_table:          (1080, 17)
metafeature_table:  (594, 114)
correlation_results: 222 features tested
correction_result:  bh
multivariate_result: None


## Correlation summary table

In [4]:
import numpy as np

# Build a comprehensive table from correlation + correction results
corr_df = pd.DataFrame([r.__dict__ for r in result.correlation_results])

if result.correction_result is not None:
    corr_df["p_value_adj"] = result.correction_result.adjusted_p_values
    corr_df["rejected"] = result.correction_result.rejected

# Add a significance star column for quick scanning
p_col = "p_value_adj" if "p_value_adj" in corr_df.columns else "p_value"
corr_df["sig"] = np.where(
    corr_df[p_col] < 0.001,
    "***",
    np.where(corr_df[p_col] < 0.01, "**", np.where(corr_df[p_col] < 0.05, "*", "")),
)

display_cols = [
    "predictor",
    "statistic",
    "ci_lower",
    "ci_upper",
    "p_value",
    *(["p_value_adj", "rejected"] if "p_value_adj" in corr_df.columns else []),
    "sig",
    "n_observations",
    "direction_confirmed",
]

corr_df[display_cols].sort_values("p_value")

,predictor,statistic,ci_lower,ci_upper,p_value,p_value_adj,rejected,sig,n_observations,direction_confirmed
168,pymfe__cls_coef,-0.488547,-0.732145,-0.130582,0.011329,0.852097,False,,26,None
128,pymfe__attr_conc.mean,0.365339,0.105932,0.579902,0.026183,0.852097,False,,37,None
208,pymfe__vdb,-0.384677,-0.669447,-0.019440,0.032616,0.852097,False,,31,None
135,pymfe__ns_ratio,-0.342817,-0.620532,0.017736,0.037791,0.852097,False,,37,None
81,pymfe__leaves_per_class.mean,0.391922,-0.006637,0.671228,0.047685,0.993388,False,,26,None
...,...,...,...,...,...,...,...,...,...,...
38,pymfe__mean.mean,-0.003077,-0.449108,0.447689,0.988098,0.993388,False,,26,None
92,pymfe__int,-0.003077,-0.434673,0.440139,0.988098,0.993388,False,,26,None
28,pymfe__cov.mean,-0.001709,-0.461099,0.460490,0.993388,0.993388,False,,26,None
49,pymfe__sd_ratio,NaN,NaN,NaN,NaN,NaN,False,,2,None


## Correlation scatter plots

In [5]:
# import matplotlib.pyplot as plt
# import seaborn as sns

# sns.set_theme(style="ticks", context="talk")

# analysis = result.analysis_table.copy()
# predictors = corr_df["predictor"].tolist()
# n_preds = len(predictors)
# ncols = min(4, n_preds)
# nrows = int(np.ceil(n_preds / ncols))

# fig, axes = plt.subplots(
#     nrows, ncols, figsize=(5.5 * ncols, 5 * nrows), sharey=True, squeeze=False
# )

# for idx, predictor in enumerate(predictors):
#     ax = axes[idx // ncols][idx % ncols]
#     plot_df = analysis[[predictor, "delta_norm"]].dropna()

#     # Use log scale for features that are strictly positive and span orders of magnitude
#     use_log = (plot_df[predictor] > 0).all() and (
#         plot_df[predictor].max() / plot_df[predictor].min() > 10
#     )

#     sns.scatterplot(data=plot_df, x=predictor, y="delta_norm", s=35, alpha=0.85, ax=ax)
#     if use_log:
#         ax.set_xscale("log")

#     ax.set_xlabel(predictor)
#     if idx % ncols == 0:
#         ax.set_ylabel(r"$\Delta_{\ell\ell}$ (NN − GBDT)")
#     else:
#         ax.set_ylabel("")

#     # Annotate with correlation and adjusted p-value
#     row = corr_df.loc[corr_df["predictor"] == predictor].iloc[0]
#     p_display = row.get("p_value_adj", row["p_value"])
#     ax.text(
#         0.05,
#         0.95,
#         f"r = {row['statistic']:.3f}\np = {p_display:.3f}",
#         transform=ax.transAxes,
#         ha="left",
#         va="top",
#         fontsize=12,
#         bbox={"facecolor": "#d9d9d9", "edgecolor": "#9e9e9e", "alpha": 0.9},
#     )

# # Hide unused subplots
# for idx in range(n_preds, nrows * ncols):
#     axes[idx // ncols][idx % ncols].set_visible(False)

# sns.despine()
# fig.tight_layout()
# plt.show()